In [1]:
import os
base_path = "/kaggle/input"
for item in os.listdir(base_path):
    print(item)

datasets


In [2]:
print(os.listdir("/kaggle/input/datasets"))

['meghdesai06']


In [3]:
print(os.listdir("/kaggle/input/datasets/meghdesai06"))

['dhristi-data']


In [4]:
print(os.listdir("/kaggle/input/datasets/meghdesai06/dhristi-data"))

['BACK CAMERA']


In [5]:
print(os.listdir("/kaggle/input/datasets/meghdesai06/dhristi-data/BACK CAMERA"))

['labels', 'images']


In [6]:
base_path = "/kaggle/input/datasets/meghdesai06/dhristi-data/BACK CAMERA"

images_path = os.path.join(base_path, "images")
labels_path = os.path.join(base_path, "labels")

images = [f for f in os.listdir(images_path) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
labels = [f for f in os.listdir(labels_path) if f.endswith(".txt")]

print("Number of images:", len(images))
print("Number of labels:", len(labels))

Number of images: 1900
Number of labels: 1900


In [7]:
base_path = "/kaggle/input/datasets/meghdesai06/dhristi-data/BACK CAMERA"
images_path = os.path.join(base_path, "images")
labels_path = os.path.join(base_path, "labels")

image_names = {
    os.path.splitext(f)[0]
    for f in os.listdir(images_path)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
}

label_names = {
    os.path.splitext(f)[0]
    for f in os.listdir(labels_path)
    if f.endswith(".txt")
}

missing_labels = image_names - label_names
missing_images = label_names - image_names

print("Images without labels:", len(missing_labels))
print("Labels without images:", len(missing_images))

Images without labels: 0
Labels without images: 0


In [8]:
import os
import random
import shutil

base_path = "/kaggle/input/datasets/meghdesai06/dhristi-data/BACK CAMERA"

images_path = os.path.join(base_path, "images")
labels_path = os.path.join(base_path, "labels")

output_path = "/kaggle/working/drishti_yolo"

# Create folders
for split in ["train", "valid", "test"]:
    os.makedirs(os.path.join(output_path, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(output_path, split, "labels"), exist_ok=True)

# Get image names
images = [
    f for f in os.listdir(images_path)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

# Shuffle consistently
random.seed(42)
random.shuffle(images)

# Split
n = len(images)

train_images = images[:int(0.8 * n)]
valid_images = images[int(0.8 * n):int(0.9 * n)]
test_images = images[int(0.9 * n):]

splits = {
    "train": train_images,
    "valid": valid_images,
    "test": test_images
}

# Copy image + matching label
for split, files in splits.items():
    for image_file in files:
        label_file = os.path.splitext(image_file)[0] + ".txt"

        shutil.copy(
            os.path.join(images_path, image_file),
            os.path.join(output_path, split, "images", image_file)
        )

        shutil.copy(
            os.path.join(labels_path, label_file),
            os.path.join(output_path, split, "labels", label_file)
        )

print("Train:", len(train_images))
print("Validation:", len(valid_images))
print("Test:", len(test_images))

Train: 1520
Validation: 190
Test: 190


In [9]:
from collections import Counter
import os

labels_path = "/kaggle/working/drishti_yolo/train/labels"

class_counts = Counter()

for file in os.listdir(labels_path):
    if file.endswith(".txt"):
        with open(os.path.join(labels_path, file), "r") as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    class_counts[int(parts[0])] += 1

print(class_counts)

Counter({5: 5427, 2: 1537, 4: 1208, 3: 1095, 1: 383, 0: 272})


In [10]:
for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file == "data.yaml":
            print(os.path.join(root, file))

In [11]:
yaml_content = """
path: /kaggle/working/drishti_yolo

train: train/images
val: valid/images
test: test/images

nc: 6

names:
  0: BackwardMove
  1: ForwardMove
  2: LeftSideMove
  3: RightSideMove
  4: Stand
  5: correct posture
"""

with open("/kaggle/working/drishti_yolo/data.yaml", "w") as f:
    f.write(yaml_content)

print("data.yaml created successfully!")

data.yaml created successfully!


In [12]:
with open("/kaggle/working/drishti_yolo/data.yaml", "r") as f:
    print(f.read())


path: /kaggle/working/drishti_yolo

train: train/images
val: valid/images
test: test/images

nc: 6

names:
  0: BackwardMove
  1: ForwardMove
  2: LeftSideMove
  3: RightSideMove
  4: Stand
  5: correct posture



In [13]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 19.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.3 MB/s eta 0:00:00


In [14]:
from ultralytics import YOLO

print("Ultralytics imported successfully")

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics imported successfully


In [15]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

print("Model loaded successfully")

Model loaded successfully


In [16]:
results = model.train(
    data="/kaggle/working/drishti_yolo/data.yaml",
    epochs=3,
    imgsz=640,
    batch=16
)

Ultralytics 8.4.123 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/drishti_yolo/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False

In [17]:
bad_file = "/kaggle/working/drishti_yolo/train/labels/602495_jpg.rf.35b8da60842c22983d6e09d59f364cc1.txt"

with open(bad_file, "r") as f:
    print(f.read())

4 0.3545416666666667 0.3531481481481481 0.06213020833333334 0.2787222222222222
0 0.05445833333333334 0.608712962962963 0.077671875 0.1796574074074074
2 0.8109583333333333 0.4140833333333333 0.059000000000000004 0.21632407407407406
2 0.875390625 0.27727777777777773 0.030473958333333332 0.08974074074074075
0 0.659375 0.3055555555555556 0.6760416666666667 0.2462962962962963 0.7114583333333333 0.17962962962962964 0.7385416666666667 0.20185185185185187 0.7416666666666667 0.24259259259259258 0.7427083333333333 0.2814814814814815 0.73125 0.32222222222222224 0.7135416666666666 0.34444444444444444 0.7052083333333333 0.34444444444444444 0.6927083333333334 0.34444444444444444 0.6625 0.3296296296296296
0 0.7444895833333334 0.19285185185185186 0.0348125 0.082
5 0.42346874999999995 0.625212962962963 0.09970833333333333 0.18736111111111112
3 0.6335416666666667 0.5492870370370371 0.08436458333333333 0.15041666666666664
3 0.728125 0.44814814814814813 0.075 0.22592592592592592


In [18]:
labels_path = "/kaggle/working/drishti_yolo/train/labels"

mixed_files = []

for file in os.listdir(labels_path):
    if file.endswith(".txt"):
        with open(os.path.join(labels_path, file), "r") as f:
            for line in f:
                parts = line.strip().split()

                if len(parts) != 5:
                    mixed_files.append(file)
                    break

print("Files containing non-standard rows:", len(mixed_files))

for file in mixed_files[:20]:
    print(file)

Files containing non-standard rows: 1
602495_jpg.rf.35b8da60842c22983d6e09d59f364cc1.txt


In [19]:
label_file = "/kaggle/working/drishti_yolo/train/labels/602495_jpg.rf.35b8da60842c22983d6e09d59f364cc1.txt"

with open(label_file, "r") as f:
    lines = f.readlines()

fixed_lines = []

for line in lines:
    parts = line.strip().split()

    if len(parts) == 5:
        # Already a normal YOLO bounding box
        fixed_lines.append(line.strip())

    else:
        # Polygon annotation
        class_id = int(parts[0])
        coords = list(map(float, parts[1:]))

        xs = coords[0::2]
        ys = coords[1::2]

        x_min = min(xs)
        x_max = max(xs)
        y_min = min(ys)
        y_max = max(ys)

        x_center = (x_min + x_max) / 2
        y_center = (y_min + y_max) / 2
        width = x_max - x_min
        height = y_max - y_min

        fixed_lines.append(
            f"{class_id} {x_center} {y_center} {width} {height}"
        )

with open(label_file, "w") as f:
    f.write("\n".join(fixed_lines) + "\n")

print("Fixed:", label_file)

Fixed: /kaggle/working/drishti_yolo/train/labels/602495_jpg.rf.35b8da60842c22983d6e09d59f364cc1.txt


In [20]:
with open(label_file, "r") as f:
    for line in f:
        print(len(line.strip().split()), line.strip())

5 4 0.3545416666666667 0.3531481481481481 0.06213020833333334 0.2787222222222222
5 0 0.05445833333333334 0.608712962962963 0.077671875 0.1796574074074074
5 2 0.8109583333333333 0.4140833333333333 0.059000000000000004 0.21632407407407406
5 2 0.875390625 0.27727777777777773 0.030473958333333332 0.08974074074074075
5 0 0.7010416666666667 0.26203703703703707 0.08333333333333326 0.1648148148148148
5 0 0.7444895833333334 0.19285185185185186 0.0348125 0.082
5 5 0.42346874999999995 0.625212962962963 0.09970833333333333 0.18736111111111112
5 3 0.6335416666666667 0.5492870370370371 0.08436458333333333 0.15041666666666664
5 3 0.728125 0.44814814814814813 0.075 0.22592592592592592


In [21]:
import os

labels_path = "/kaggle/working/drishti_yolo/train/labels"

bad_files = []

for file in os.listdir(labels_path):
    if file.endswith(".txt"):
        with open(os.path.join(labels_path, file), "r") as f:
            for line in f:
                if len(line.strip().split()) != 5:
                    bad_files.append(file)
                    break

print("Files with invalid rows:", len(bad_files))

Files with invalid rows: 0


In [22]:
results = model.train(
    data="/kaggle/working/drishti_yolo/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    project="/kaggle/working/runs",
    name="drishti_yolov8n"
)

Ultralytics 8.4.123 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/drishti_yolo/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/working/runs/detect/train/weights/best.pt, momentum=0.937, mosaic=1.0, multi_